# Train Event Encoder V2 on Colab GPU

This notebook clones the repo, mounts Google Drive, and runs `train_event_encoder_v2.py` with GPU-friendly settings.

Before running, set the Colab runtime to `GPU` from `Runtime > Change runtime type`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Switch the Colab runtime to GPU before training.')


In [ ]:
from pathlib import Path

# Update these if your repo or branch changes.
REPO_URL = 'https://github.com/thomas-5/MLSE-PlayerBERT.git'
BRANCH = 'feature_engineering'
REPO_DIR = Path('/content/MLSE-PlayerBERT')

# Google Drive paths.
DATA_PATH = Path('/content/drive/MyDrive/MLSE/events360_v4.jsonl')
OUTPUT_PATH = Path('/content/drive/MyDrive/MLSE/models/event_encoder_v2.pt')

# Training settings.
EPOCHS = 3
BATCH_SIZE = 64
MAX_ROWS = None          # Set to e.g. 200000 for a quicker smoke run
NUM_WORKERS = 2
LR = 1e-3
D_MODEL = 192
OUT_DIM = 128
NUM_HEADS = 6
EVENT_LAYERS = 2
FRAME_LAYERS = 2
USE_AMP = True


In [ ]:
import os

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    %cd /content
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO_DIR}

!python3 -m pip install -q --upgrade pip
!python3 -m pip install -q tqdm


In [ ]:
assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('DATA_PATH:', DATA_PATH)
print('OUTPUT_PATH:', OUTPUT_PATH)


In [ ]:
import subprocess

help_result = subprocess.run(
    ['python3', 'train_event_encoder_v2.py', '--help'],
    text=True,
    capture_output=True,
)
help_text = help_result.stdout + '\n' + help_result.stderr
supports_num_workers = '--num_workers' in help_text
supports_amp = '--amp' in help_text

print('supports_num_workers:', supports_num_workers)
print('supports_amp:', supports_amp)

cmd = [
    'python3', 'train_event_encoder_v2.py',
    '--data_path', str(DATA_PATH),
    '--output', str(OUTPUT_PATH),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--d_model', str(D_MODEL),
    '--out_dim', str(OUT_DIM),
    '--num_heads', str(NUM_HEADS),
    '--event_layers', str(EVENT_LAYERS),
    '--frame_layers', str(FRAME_LAYERS),
]
if supports_num_workers:
    cmd += ['--num_workers', str(NUM_WORKERS)]
if MAX_ROWS is not None:
    cmd += ['--max_rows', str(MAX_ROWS)]
if USE_AMP and supports_amp:
    cmd += ['--amp']

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Training failed with return code {result.returncode}')


In [ ]:
import torch

print('Checkpoint written to:', OUTPUT_PATH)
print('Exists:', OUTPUT_PATH.exists())

if OUTPUT_PATH.exists():
    ckpt = torch.load(OUTPUT_PATH, map_location='cpu', weights_only=False)
    print('Checkpoint keys:', list(ckpt.keys()))
    print('Feature count:', len(ckpt['features']))
    print('Args used:', ckpt['args'])
else:
    print('No checkpoint found because training did not complete successfully.')
